### Data Ingestion

In [1]:
from langchain_core.documents import Document

In [2]:
#creating an eg document
doc1=Document(
    page_content="Hi my name is Abhinav, what's your name? I'm glad to meet you",
    metadata={
        "source":"example.txt",
        "page_no":1,
        "autor":"Abhinav Nagathan"
    }
)

doc1

Document(metadata={'source': 'example.txt', 'page_no': 1, 'autor': 'Abhinav Nagathan'}, page_content="Hi my name is Abhinav, what's your name? I'm glad to meet you")

In [3]:
#creating a simple txt file
import os
os.makedirs("../data/text_files",exist_ok=True)

In [4]:
sample1={"../data/text_files/ai_intro.txt":""" Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence.
It involves technologies such as machine learning, natural language processing, and computer vision.
AI systems can analyze large amounts of data, recognize patterns, and make decisions.
These systems are used in many areas including healthcare, education, finance, and transportation.
Examples of AI applications include virtual assistants, recommendation systems, and self-driving cars.
AI helps improve efficiency, accuracy, and automation in many industries.
It also supports problem-solving and predictive analysis.
As technology advances, AI continues to become more powerful and widely used.
However, it also raises important ethical and social considerations.
Overall, AI is transforming the way humans interact with technology and the world.
""",
"../data/text_files/space_exploration_latest.txt": """ 
Space exploration is the scientific study and discovery of outer space using advanced technology and spacecraft.
In recent years, many new missions have expanded our understanding of the universe.
Modern space exploration involves rockets, satellites, space telescopes, and robotic probes.
Scientists use these technologies to study planets, stars, galaxies, and other cosmic phenomena.
Recent missions have explored Mars, collected asteroid samples, and observed distant galaxies.
Private space companies are also contributing by developing reusable rockets and planning human missions to the Moon and Mars.
Space telescopes are capturing detailed images that help scientists understand the origins of the universe.
Researchers are also searching for signs of water and possible life on other planets and moons.
These discoveries help improve our knowledge of space, planetary systems, and Earth's place in the universe.
However, space exploration also presents challenges such as high costs, technical complexity, and space debris.
Overall, the latest exploration in space is expanding human knowledge and opening new possibilities for future space travel and discovery.
"""
}

for filepath, content in sample1.items():
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(content)

#### Reading the text using TextLoader

In [5]:
#textloader

from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/ai_intro.txt",encoding='utf-8',autodetect_encoding=True)

new_doc=loader.load()
print(new_doc)

#here we see that the metadata has also been updated

e:\ABHINAV\Coding\Learning\Rag_learn\.myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/ai_intro.txt'}, page_content=' Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence.\nIt involves technologies such as machine learning, natural language processing, and computer vision.\nAI systems can analyze large amounts of data, recognize patterns, and make decisions.\nThese systems are used in many areas including healthcare, education, finance, and transportation.\nExamples of AI applications include virtual assistants, recommendation systems, and self-driving cars.\nAI helps improve efficiency, accuracy, and automation in many industries.\nIt also supports problem-solving and predictive analysis.\nAs technology advances, AI continues to become more powerful and widely used.\nHowever, it also raises important ethical and social considerations.\nOverall, AI is transforming the way humans interact with technology and the w

In [19]:
#directory loader

from langchain_community.document_loaders import DirectoryLoader

#loading all the files form data dir
dir_loader=DirectoryLoader("../data/text_files",
                           glob="**/*.txt",
                           loader_cls=TextLoader,
                           )
#A glob pattern or list of glob patterns to use to find files.  Defaults to "**/[!.]*" (all files except hidden).

dir_files=dir_loader.load()
dir_files

[Document(metadata={'source': '..\\data\\text_files\\ai_intro.txt'}, page_content=' Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence.\nIt involves technologies such as machine learning, natural language processing, and computer vision.\nAI systems can analyze large amounts of data, recognize patterns, and make decisions.\nThese systems are used in many areas including healthcare, education, finance, and transportation.\nExamples of AI applications include virtual assistants, recommendation systems, and self-driving cars.\nAI helps improve efficiency, accuracy, and automation in many industries.\nIt also supports problem-solving and predictive analysis.\nAs technology advances, AI continues to become more powerful and widely used.\nHowever, it also raises important ethical and social considerations.\nOverall, AI is transforming the way humans interact with technology and th

In [7]:
#for pdf's -> load the pdf's into 

#pymupdf is better when compared to pypdf

from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

dir_loader=DirectoryLoader("../data/pdfs",
                           glob="**/*.pdf",
                           loader_cls=PyMuPDFLoader
                           )


pdf_files=dir_loader.load()
for i in pdf_files:
    print(i)

page_content='Unit- I
Introduction' metadata={'producer': 'Microsoft® PowerPoint® 2019', 'creator': 'Microsoft® PowerPoint® 2019', 'creationdate': '2025-10-01T20:54:48+05:30', 'source': '..\\data\\pdfs\\Mod1_RE.pdf', 'file_path': '..\\data\\pdfs\\Mod1_RE.pdf', 'total_pages': 76, 'format': 'PDF 1.7', 'title': '', 'author': 'Naseem Khayum', 'subject': '', 'keywords': '', 'moddate': '2025-11-01T23:48:55+05:30', 'trapped': '', 'modDate': "D:20251101234855+05'30'", 'creationDate': "D:20251001205448+05'30'", 'page': 0}
page_content='Source: IPCC2021;Summary of policy makers
Human Impact on Climate Change' metadata={'producer': 'Microsoft® PowerPoint® 2019', 'creator': 'Microsoft® PowerPoint® 2019', 'creationdate': '2025-10-01T20:54:48+05:30', 'source': '..\\data\\pdfs\\Mod1_RE.pdf', 'file_path': '..\\data\\pdfs\\Mod1_RE.pdf', 'total_pages': 76, 'format': 'PDF 1.7', 'title': '', 'author': 'Naseem Khayum', 'subject': '', 'keywords': '', 'moddate': '2025-11-01T23:48:55+05:30', 'trapped': '', 'm

### Chunking

In [8]:
# Creating Data Chunks 

from langchain_text_splitters import RecursiveCharacterTextSplitter

### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    #print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    #if split_docs:
    #    print(f"\nExample chunk:")
    #    print(f"Content: {split_docs[0].page_content[:200]}...")
    #    print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks=split_documents(dir_files)
chunks

[Document(metadata={'source': '..\\data\\text_files\\ai_intro.txt'}, page_content='Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence.\nIt involves technologies such as machine learning, natural language processing, and computer vision.\nAI systems can analyze large amounts of data, recognize patterns, and make decisions.\nThese systems are used in many areas including healthcare, education, finance, and transportation.\nExamples of AI applications include virtual assistants, recommendation systems, and self-driving cars.\nAI helps improve efficiency, accuracy, and automation in many industries.\nIt also supports problem-solving and predictive analysis.\nAs technology advances, AI continues to become more powerful and widely used.\nHowever, it also raises important ethical and social considerations.\nOverall, AI is transforming the way humans interact with technology and the

### Embedding and vector DB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid  
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
#convers sentences to vectors

class EmbeddingManager:

    #here we're initalizing the embeddng manager
    def __init__(self,model_name: str="all-MiniLM-L6-v2"):      #model_name= all-Minilm-l6-v2 is a HF model for sentence embedding
        self.model_name=model_name
        self.model=None
        self._load_model() #its function is to load all-Minilm-l6-v2

    def _load_model(self):

        try:
            print("Loading the Model:",self.model_name)
            self.model=SentenceTransformer(self.model_name)
            print(f"\nModel loaded sucessfully and its dimmensions are: {self.model.get_sentence_embedding_dimension()}") #Returns the number of dimensions in the output of SentenceTransformer.encode
        
        except Exception as e:
            print("Error while loading the model:",e)
            raise
    

    #takes texts i.e strings then convers it to numpy arrays
    def generate_embeddings(self, texts: List[str])-> np.ndarray:
        if not self.model:
            raise ValueError("Model is not loaded")
        
        embeddings=self.model.encode(texts, show_progress_bar=True) #here we are calling .encode() manually and in the below cell, we're using semantic chunking and it cant be called since it uses different methods, hence we use a LC wrapper=HF
        print("Embeddings generated sucessfully.")
        return embeddings


#initalizing 
embedding_manager=EmbeddingManager()
embedding_manager


Loading the Model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3121.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model loaded sucessfully and its dimmensions are: 384


#### Update: There are different types of chunking mechanisms: 
•	semantic chunking 
•	overlap windows 
•	sentence-aware splits 


In [ ]:
#Semantic chunking

#pip install langchain-experimental 

from langchain_experimental.text_splitter import SemanticChunker

def split_doc_semantic(documents, embedding_manager):
    
    #Segmenting the text
    text_splitter= SemanticChunker(embedding_manager.model)     #basically reusing the exiting code, i.e Loading the Model: all-MiniLM-L6-v2
    split_docs=text_splitter.split_documents(documents)

    return split_docs

#chunks=split_doc_semantic(dir_files,embedding_manager)
#print(chunks)

#error occurs: 'SentenceTransformer' object has no attribute 'embed_documents', i.e we cannot pass embedding_manager.model as there is no attribute 

In [ ]:
#Semantic chunking
#pip install langchain-experimental 


from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

def split_doc_semantic(documents):
    
    #import the embedding model
    
    #embedding_model=SentenceTransformer("all-MiniLM-L6-v2")  -> using this gives 'SentenceTransformer' object has no attribute 'embed_documents' because
    # LangChain's SemanticChunker expects an object which uses its own methods (embed_documents and embed_query)  
    # and the raw SentenceTransformer object from the sentence-transformers library uses a method named .encode(), which is why the code is crashing.

    # This wrapper makes the model compatible with RAG tools
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    #Segmenting the text
    text_splitter= SemanticChunker(embedding_model)     
    splitted_docs=text_splitter.split_documents(documents)

    print(f"Total chunks created: {len(splitted_docs)}")
    return splitted_docs

chunks=split_doc_semantic(dir_files)
print(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1320.20it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total chunks created: 4
[Document(metadata={'source': '..\\data\\text_files\\ai_intro.txt'}, page_content=' Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence. It involves technologies such as machine learning, natural language processing, and computer vision. AI systems can analyze large amounts of data, recognize patterns, and make decisions. These systems are used in many areas including healthcare, education, finance, and transportation. Examples of AI applications include virtual assistants, recommendation systems, and self-driving cars. AI helps improve efficiency, accuracy, and automation in many industries.'), Document(metadata={'source': '..\\data\\text_files\\ai_intro.txt'}, page_content='It also supports problem-solving and predictive analysis. As technology advances, AI continues to become more powerful and widely used. However, it also raises important ethical a

### VectorStore -> vectore storage with chromaDB

In [23]:
#chromadb storage for the embeddings


#collection_name-> is the name of chromadb collection
#persist_directory-> direcotry where to store vectors 

class VectorStore:
    def __init__(self, collection_name:str="pdf_documents" , persist_directory:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self.initalize_store()

    #initalize chromadb client and colection
    def initalize_store(self):

        try:
            #create persistent chromadb client that has acess to chromadb storage
            os.makedirs(self.persist_directory, exist_ok= True)
            #creating a connection/DB(if already exist) to chromadb client by giving it the storage address and 
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            #get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                
                )
            print("Sucessfully initalised vector store")

        except Exception as e:
            print(f"Error occured while initalizing vector store: {e}")
            raise #raise = Error happens → program stops immediately ✅ ; w/o raise: Error happens → printed → program continues ❌


    #adding documents and its embd to vectors        
    def add_docs(self, documents:List[any],embeddings:np.ndarray):

        if len(documents)!=len(embeddings):
            print("Number of documents is not same as embeddings")
            raise

        print(f"Adding {len(documents)} documents to vector storage..")

        #prepare data for chromadb
        ids=[]
        metadatas=[]
        document_text=[]
        embedding_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):

            #generate unique id
            doc_id= f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #prepare metadata
            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['context_len']= len(doc.page_content)
            metadatas.append(metadata)

            #document text
            doc_text=doc.page_content
            document_text.append(doc_text)

            #embedding appended to embedding_list
            embedding_list.append(embedding.tolist())

            #add this data to collection
        try:
            self.collection.add(
                    ids=ids,
                    embeddings=embedding_list,
                    metadatas=metadatas,
                    documents=document_text
                )
            print(f"sucessfully added {len(documents)} to vector store and total collections are {self.collection.count()}")


        except Exception as e:
            print("Error encountered while adding documents and its embd to vectorstore: ",e)
            raise

    
#initalization
vector_store=VectorStore()
vector_store

Sucessfully initalised vector store


In [24]:
#refer cell of markdown "Chunking"
chunks

[Document(metadata={'source': '..\\data\\text_files\\ai_intro.txt'}, page_content=' Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines capable of performing tasks that normally require human intelligence. It involves technologies such as machine learning, natural language processing, and computer vision. AI systems can analyze large amounts of data, recognize patterns, and make decisions. These systems are used in many areas including healthcare, education, finance, and transportation. Examples of AI applications include virtual assistants, recommendation systems, and self-driving cars. AI helps improve efficiency, accuracy, and automation in many industries.'),
 Document(metadata={'source': '..\\data\\text_files\\ai_intro.txt'}, page_content='It also supports problem-solving and predictive analysis. As technology advances, AI continues to become more powerful and widely used. However, it also raises important ethical and social consideration

In [25]:
#converting text to embeddings
texts= [doc.page_content for doc in chunks]
texts

#genrating the embeddings from texts which gotten from chunks

#for this we use embedding manager we created and use its .generate_embeddings

embeddings= embedding_manager.generate_embeddings(texts)


#store the embeddings into vecotr db using .add_docs
storage=vector_store.add_docs(documents=chunks,embeddings=embeddings)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.01it/s]

Embeddings generated sucessfully.
Adding 4 documents to vector storage..
sucessfully added 4 to vector store and total collections are 631


### Reteriver Pipeline from VectorStore

In [26]:
#handles Query based retriveal from the vector store
#built on top of Vstore
#simple interface that gives data from vector storage based on user query
class RAGRetriver:

    def __init__(self,vector_store:VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store=vector_store
        self.embedding_manager = embedding_manager
    
    #top_k is the amout of results we want
    #it returs a list of result
    def retrive(self, query:str, top_k:int=5, score_threshold:float=0.0) -> List[Dict[str , Any]]:
        
        #converting quesry to embedding
        query_embedding=self.embedding_manager.generate_embeddings([query])[0] 

        #[0] Because the function returns:[[vector]] and we extract the first one only

        #search in vector store
        try:
            results=self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            #Process the results
            retrived_docs=[]

            if results['documents'] and results['documents'][0]:
                documents=results['documents'][0]
                metadatas=results['metadatas'][0]
                distances=results['distances'][0]
                ids=results['ids'][0]

                for i,(doc_id,document,metadata,distance) in enumerate (zip(ids,documents,metadatas,distances)):

                    #convert the distance to similarity score
                    similarityScore=1-distance
                    
                    if similarityScore>=score_threshold:
                        retrived_docs.append({
                            'id':doc_id,
                            'content':document,
                            'metadata':metadata,
                            'distance':distance,
                            'similarity_score':similarityScore,
                            'rank':i+1
                        })

                print(f"Reterived {len(retrived_docs)} number of documents after filtering")
            
            else:
                print("No documents found.")

            return retrived_docs
        
        except Exception as e:
            print(f"Error during reterivl: {e}")
            return []

        
#creating an object of this class
rag_retriver=RAGRetriver(vector_store,embedding_manager)
rag_retriver

#the o/p shows that it is an objec of this class


In [27]:
#using the reteriver object

rag_retriver.retrive("What is renewable energy?")
#what this is doing, in short, gives all the reterived docs/ context which the "retrive" fn does, we can use it more eff to automatically feed it to get output from LLM

Batches: 100%|██████████| 1/1 [00:00<00:00, 45.46it/s]

Embeddings generated sucessfully.
Reterived 5 number of documents after filtering


[{'id': 'doc_25913a1a_11',
  'content': 'Fig. 7 Significance of Renewable Energy',
  'metadata': {'trapped': '',
   'creator': 'Microsoft® PowerPoint® 2019',
   'moddate': '2025-11-01T23:48:55+05:30',
   'creationDate': "D:20251001205448+05'30'",
   'subject': '',
   'total_pages': 76,
   'context_len': 39,
   'file_path': '..\\data\\pdfs\\Mod1_RE.pdf',
   'doc_index': 11,
   'producer': 'Microsoft® PowerPoint® 2019',
   'format': 'PDF 1.7',
   'creationdate': '2025-10-01T20:54:48+05:30',
   'modDate': "D:20251101234855+05'30'",
   'author': 'Naseem Khayum',
   'source': '..\\data\\pdfs\\Mod1_RE.pdf',
   'title': '',
   'page': 11,
   'keywords': ''},
  'distance': 0.4749232828617096,
  'similarity_score': 0.5250767171382904,
  'rank': 1},
 {'id': 'doc_6f1c4544_11',
  'content': 'Fig. 7 Significance of Renewable Energy',
  'metadata': {'total_pages': 76,
   'moddate': '2025-11-01T23:48:55+05:30',
   'trapped': '',
   'creator': 'Microsoft® PowerPoint® 2019',
   'page': 11,
   'author':

### Generation Layer: Integration of context and User Query to a LLM to get results

In [28]:
#simple RAG peipeline with groqllm

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

#initalize groqllm
groq_api_key=os.getenv("RAG_LEARN")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

#simple rag fn: retrive context +generate response
def rag_pipeline(query:str,retriver,llm,top_k:int=3):

    #retrive the context
    results=retriver.retrive(query,top_k)

    #combine to get context
    if results:
        context="\n\n".join([doc['content'] for doc in results])  
    else:
        context=""
    
    if not context: 
        return "No context was found." 
    
    #generate the answer using groqLLM
    prompt = f"""
    Answer the question using the context below:

    Context:
    {context}

    Question:
    {query}

    Answer:
    """ 

    response = llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [32]:
answer= rag_pipeline("Explain Recent missions have explored Mars in 50 words",rag_retriver,llm)
print(answer)

Batches: 100%|██████████| 1/1 [00:00<00:00, 23.26it/s]


Embeddings generated sucessfully.
Reterived 3 number of documents after filtering
Recent missions have significantly expanded our understanding of Mars, including NASA's Curiosity Rover, which has been exploring the planet since 2012, and the Perseverance Rover, which landed in 2021 to search for signs of past or present life on the Red Planet.
